In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from IPython.display import HTML


def media_geometrica_assinada(vetor):
    """
    Calcula uma extensão assinada da média geométrica para valores reais.

    A média geométrica clássica exige valores positivos. Para permitir valores
    negativos, usamos:

        G = sign(prod(x_i)) * exp(mean(log(abs(x_i))))

    Se algum valor for zero, retorna zero.
    """

    vetor = np.asarray(vetor, dtype=float)

    if np.any(vetor == 0):
        return 0.0

    sinal = np.prod(np.sign(vetor))
    modulo = np.exp(np.mean(np.log(np.abs(vetor))))

    return sinal * modulo


def media_harmonica_real(vetor):
    """
    Calcula a média harmônica para valores reais não nulos.

        H = n / sum(1 / x_i)

    Se algum valor for zero, retorna NaN.
    Se a soma dos recíprocos for aproximadamente zero, retorna NaN.
    """

    vetor = np.asarray(vetor, dtype=float)

    if np.any(vetor == 0):
        return np.nan

    soma_reciprocos = np.sum(1.0 / vetor)

    if np.isclose(soma_reciprocos, 0.0):
        return np.nan

    return len(vetor) / soma_reciprocos


def calcular_medias_pontos(pontos):
    """
    Calcula médias componente a componente para pontos 2D.

    Para pontos p_i = (x_i, y_i), calcula:
        - média aritmética;
        - média geométrica assinada;
        - média harmônica real;
        - média quadrática, isto é, RMS.
    """

    pontos = np.asarray(pontos, dtype=float)

    x = pontos[:, 0]
    y = pontos[:, 1]

    media_aritmetica = np.array([
        np.mean(x),
        np.mean(y)
    ])

    media_geometrica = np.array([
        media_geometrica_assinada(x),
        media_geometrica_assinada(y)
    ])

    media_harmonica = np.array([
        media_harmonica_real(x),
        media_harmonica_real(y)
    ])

    media_quadratica = np.array([
        np.sqrt(np.mean(x**2)),
        np.sqrt(np.mean(y**2))
    ])

    return {
        "Aritmética": media_aritmetica,
        "Geométrica assinada": media_geometrica,
        "Harmônica": media_harmonica,
        "Quadrática": media_quadratica,
    }


def criar_dataframe_pontos_medias(pontos, medias):
    """
    Cria um DataFrame contendo os pontos originais e as médias calculadas.
    """

    registros = []

    # Registra os pontos aleatórios.
    for i, ponto in enumerate(pontos, start=1):
        registros.append({
            "tipo": "ponto",
            "nome": f"Ponto {i}",
            "indice": i,
            "x": ponto[0],
            "y": ponto[1],
        })

    # Registra as médias.
    for nome_media, valor_media in medias.items():
        registros.append({
            "tipo": "media",
            "nome": nome_media,
            "indice": np.nan,
            "x": valor_media[0],
            "y": valor_media[1],
        })

    return pd.DataFrame(registros)


def obter_figsize_por_formato(
    formato="1:1",
    largura_base=8.0,
    figsize=None
):
    """
    Define o tamanho da figura a partir do formato visual.

    Parâmetros
    ----------
    formato : str
        Formato da animação:
            - "1:1"
            - "16:9"
            - "9:16"

    largura_base : float
        Largura da figura em polegadas.

    figsize : tuple ou None
        Tamanho explícito da figura. Se fornecido, tem prioridade.
    """

    if figsize is not None:
        return figsize

    if formato == "1:1":
        return (largura_base, largura_base)

    if formato == "16:9":
        return (largura_base, largura_base * 9 / 16)

    if formato == "9:16":
        return (largura_base, largura_base * 16 / 9)

    raise ValueError(
        "Formato inválido. Use '1:1', '16:9', '9:16' ou forneça figsize."
    )


def animar_medias_2d(
    n_pontos=20,
    x_range=(-10.0, 10.0),
    y_range=(-10.0, 10.0),
    x_grid_range=None,
    y_grid_range=None,
    grid_step=None,
    seed=42,
    intervalo=1000,
    pausa_final_segundos=5,
    salvar_dataset_como=None,
    salvar_animacao_como=None,
    formato="1:1",
    largura_base=8.0,
    figsize=None,
    dpi=120,
    fps_export=None,
    tamanho_pontos=20,
    alpha_pontos=0.50,
    tamanho_medias=110,
):
    """
    Cria uma animação 2D com pontos aleatórios e médias vetoriais.

    Parâmetros principais
    ---------------------
    n_pontos : int
        Quantidade de pontos aleatórios.

    x_range, y_range : tuple
        Intervalos usados para gerar os pontos.
        Exemplo: x_range=(-10, 10), y_range=(-8, 12).

    x_grid_range, y_grid_range : tuple ou None
        Limites visuais do gráfico.
        Se None, são definidos automaticamente a partir dos pontos e médias.

    grid_step : float, tuple ou None
        Espaçamento do grid.
        Exemplo:
            grid_step=5
            grid_step=(5, 2)

    formato : str
        Formato visual da animação:
            - "1:1"
            - "16:9"
            - "9:16"

    Retorna
    -------
    df_dataset : pd.DataFrame
        DataFrame com pontos e médias.

    anim : matplotlib.animation.FuncAnimation
        Objeto da animação.
    """

    # ------------------------------------------------------------
    # Validações iniciais
    # ------------------------------------------------------------

    x_min, x_max = x_range
    y_min, y_max = y_range

    if n_pontos <= 0:
        raise ValueError("n_pontos deve ser maior que zero.")

    if x_min >= x_max:
        raise ValueError("x_range deve ser da forma (x_min, x_max), com x_min < x_max.")

    if y_min >= y_max:
        raise ValueError("y_range deve ser da forma (y_min, y_max), com y_min < y_max.")

    if intervalo <= 0:
        raise ValueError("intervalo deve ser maior que zero.")

    if pausa_final_segundos < 0:
        raise ValueError("pausa_final_segundos deve ser maior ou igual a zero.")

    # ------------------------------------------------------------
    # Define tamanho da figura
    # ------------------------------------------------------------

    figsize_final = obter_figsize_por_formato(
        formato=formato,
        largura_base=largura_base,
        figsize=figsize
    )

    # ------------------------------------------------------------
    # Gera pontos aleatórios
    # ------------------------------------------------------------

    rng = np.random.default_rng(seed)

    pontos = np.column_stack([
        rng.uniform(x_min, x_max, n_pontos),
        rng.uniform(y_min, y_max, n_pontos),
    ])

    # ------------------------------------------------------------
    # Calcula médias
    # ------------------------------------------------------------

    medias = calcular_medias_pontos(pontos)

    # ------------------------------------------------------------
    # Cria DataFrame
    # ------------------------------------------------------------

    df_dataset = criar_dataframe_pontos_medias(pontos, medias)

    # O dataset não é salvo automaticamente.
    # Só salva se o usuário passar explicitamente um caminho.
    if salvar_dataset_como is not None:
        df_dataset.to_csv(salvar_dataset_como, index=False)

    # ------------------------------------------------------------
    # Cores das médias
    # ------------------------------------------------------------

    cores_medias = {
        "Aritmética": "blue",
        "Geométrica assinada": "green",
        "Harmônica": "orange",
        "Quadrática": "red",
    }

    nomes_medias = list(medias.keys())

    # ------------------------------------------------------------
    # Define os frames da animação
    # ------------------------------------------------------------

    n_frames_pontos = n_pontos
    n_frames_medias = len(nomes_medias)
    total_frames = n_frames_pontos + n_frames_medias

    # Repete o último frame para produzir uma pausa final.
    pausa_final_frames = int(
        np.ceil(1000 * pausa_final_segundos / intervalo)
    )

    frames_animacao = (
        list(range(total_frames))
        + [total_frames - 1] * pausa_final_frames
    )

    # ------------------------------------------------------------
    # Cria figura
    # ------------------------------------------------------------

    fig, ax = plt.subplots(figsize=figsize_final)

    # ------------------------------------------------------------
    # Define limites visuais do grid
    # ------------------------------------------------------------

    medias_x = np.array([m[0] for m in medias.values()], dtype=float)
    medias_y = np.array([m[1] for m in medias.values()], dtype=float)

    # Caso o usuário não defina x_grid_range, calcula automaticamente.
    if x_grid_range is None:
        x_abs_max = max(
            abs(x_min),
            abs(x_max),
            np.max(np.abs(pontos[:, 0])),
            np.nanmax(np.abs(medias_x)),
            1.0
        )
        x_grid_range = (-1.20 * x_abs_max, 1.20 * x_abs_max)

    # Caso o usuário não defina y_grid_range, calcula automaticamente.
    if y_grid_range is None:
        y_abs_max = max(
            abs(y_min),
            abs(y_max),
            np.max(np.abs(pontos[:, 1])),
            np.nanmax(np.abs(medias_y)),
            1.0
        )
        y_grid_range = (-1.20 * y_abs_max, 1.20 * y_abs_max)

    x_grid_min, x_grid_max = x_grid_range
    y_grid_min, y_grid_max = y_grid_range

    if x_grid_min >= x_grid_max:
        raise ValueError(
            "x_grid_range deve ser da forma (x_min, x_max), com x_min < x_max."
        )

    if y_grid_min >= y_grid_max:
        raise ValueError(
            "y_grid_range deve ser da forma (y_min, y_max), com y_min < y_max."
        )

    # Validação do espaçamento do grid.
    if grid_step is not None:
        if isinstance(grid_step, tuple):
            grid_step_x, grid_step_y = grid_step
        else:
            grid_step_x = grid_step
            grid_step_y = grid_step

        if grid_step_x <= 0 or grid_step_y <= 0:
            raise ValueError("grid_step deve ser positivo.")
    else:
        grid_step_x = None
        grid_step_y = None

    # ------------------------------------------------------------
    # Funções internas de desenho
    # ------------------------------------------------------------

    def configurar_eixos():
        """
        Configura os eixos e o grid a cada frame.
        """

        ax.set_xlim(x_grid_min, x_grid_max)
        ax.set_ylim(y_grid_min, y_grid_max)

        # Mantém a mesma escala nos dois eixos.
        ax.set_aspect("equal", adjustable="box")

        # Eixos cartesianos.
        ax.axhline(0, color="black", linewidth=1.0, alpha=0.6)
        ax.axvline(0, color="black", linewidth=1.0, alpha=0.6)

        # Ticks parametrizados do grid.
        if grid_step_x is not None and grid_step_y is not None:
            ax.set_xticks(
                np.arange(x_grid_min, x_grid_max + grid_step_x, grid_step_x)
            )
            ax.set_yticks(
                np.arange(y_grid_min, y_grid_max + grid_step_y, grid_step_y)
            )

        ax.grid(True, alpha=0.3)

        ax.set_xlabel(r"$x$")
        ax.set_ylabel(r"$y$")
        ax.set_title("Pontos aleatórios e médias vetoriais no plano")

        # Origem marcada discretamente, sem texto.
        ax.scatter(
            0,
            0,
            color="black",
            s=18,
            zorder=5,
            alpha=0.8
        )

    def desenhar_vetor(destino, cor="black", alpha=1.0, lw=1.5):
        """
        Desenha um vetor partindo da origem até o ponto destino.
        """

        destino = np.asarray(destino, dtype=float)

        # Não desenha vetor se houver coordenada indefinida.
        if np.any(np.isnan(destino)):
            return

        ax.annotate(
            "",
            xy=destino,
            xytext=(0, 0),
            arrowprops=dict(
                arrowstyle="->",
                color=cor,
                lw=lw,
                alpha=alpha,
                shrinkA=0,
                shrinkB=0,
            ),
        )

    def update(frame):
        """
        Atualiza cada frame da animação.
        """

        ax.clear()
        configurar_eixos()

        # --------------------------------------------------------
        # Fase 1: pontos aparecem progressivamente
        # --------------------------------------------------------

        if frame < n_frames_pontos:
            n_visiveis = frame + 1
            pontos_visiveis = pontos[:n_visiveis]

            # Vetores dos pontos, discretos.
            for p in pontos_visiveis:
                desenhar_vetor(
                    p,
                    cor="black",
                    alpha=0.30,
                    lw=0.9
                )

            # Pontos pequenos e sutis, sem legenda.
            ax.scatter(
                pontos_visiveis[:, 0],
                pontos_visiveis[:, 1],
                color="black",
                s=tamanho_pontos,
                alpha=alpha_pontos,
                zorder=10
            )

        # --------------------------------------------------------
        # Fase 2: médias aparecem progressivamente
        # --------------------------------------------------------

        else:
            # Vetores dos pontos originais, ainda mais discretos.
            for p in pontos:
                desenhar_vetor(
                    p,
                    cor="black",
                    alpha=0.18,
                    lw=0.8
                )

            # Pontos originais, sem legenda.
            ax.scatter(
                pontos[:, 0],
                pontos[:, 1],
                color="black",
                s=tamanho_pontos,
                alpha=alpha_pontos,
                zorder=10
            )

            # Número de médias visíveis no frame atual.
            n_medias_visiveis = frame - n_frames_pontos + 1

            for nome in nomes_medias[:n_medias_visiveis]:
                media = medias[nome]
                cor = cores_medias[nome]

                # Pula médias indefinidas, como harmônica singular.
                if np.any(np.isnan(media)):
                    continue

                # Vetor da média.
                desenhar_vetor(
                    media,
                    cor=cor,
                    alpha=1.0,
                    lw=2.2
                )

                # Ponto da média.
                # O label aparece apenas na legenda.
                ax.scatter(
                    media[0],
                    media[1],
                    color=cor,
                    s=tamanho_medias,
                    edgecolor="black",
                    zorder=20,
                    label=nome
                )

        # Legenda apenas das médias.
        handles, labels = ax.get_legend_handles_labels()
        if len(handles) > 0:
            ax.legend(loc="upper left")

        return ax,

    # ------------------------------------------------------------
    # Cria animação
    # ------------------------------------------------------------

    anim = FuncAnimation(
        fig,
        update,
        frames=frames_animacao,
        interval=intervalo,
        blit=False,
        repeat=False
    )

    # ------------------------------------------------------------
    # Define FPS para exportação
    # ------------------------------------------------------------

    if fps_export is None:
        fps_export = max(1, int(round(1000.0 / intervalo)))

    # ------------------------------------------------------------
    # Salva animação apenas se solicitado
    # ------------------------------------------------------------

    if salvar_animacao_como is not None:
        if salvar_animacao_como.lower().endswith(".gif"):
            anim.save(
                salvar_animacao_como,
                writer=PillowWriter(fps=fps_export),
                dpi=dpi
            )

        elif salvar_animacao_como.lower().endswith(".mp4"):
            anim.save(
                salvar_animacao_como,
                writer=FFMpegWriter(fps=fps_export, bitrate=1800),
                dpi=dpi
            )

        else:
            anim.save(salvar_animacao_como, dpi=dpi)

    # Fecha a figura estática para não aparecer automaticamente no Jupyter.
    plt.close(fig)

    return df_dataset, anim

In [2]:
df, anim = animar_medias_2d(
    n_pontos=10,
    x_range=(-15, 15),
    y_range=(-15, 15),
    x_grid_range=(-20, 20),
    y_grid_range=(-20, 20),
    grid_step=5,
    seed=7,
    intervalo=1000,
    pausa_final_segundos=6,
    formato="16:9",
    largura_base=10,
    tamanho_pontos=20,
    alpha_pontos=0.45,
    tamanho_medias=30,
    salvar_dataset_como=None,
    salvar_animacao_como=None
)

df

,tipo,nome,indice,x,y
0,ponto,Ponto 1,1.0,3.752864,-5.909027
1,ponto,Ponto 2,2.0,11.916414,-6.647232
2,ponto,Ponto 3,3.0,8.270571,-7.353912
3,ponto,Ponto 4,4.0,-8.243784,-1.647711
4,ponto,Ponto 5,5.0,-5.995011,0.136448
5,ponto,Ponto 6,6.0,11.206603,1.604921
6,ponto,Ponto 7,7.0,-14.842041,14.865009
7,ponto,Ponto 8,8.0,9.636853,8.779858
8,ponto,Ponto 9,9.0,8.912083,3.665377
9,ponto,Ponto 10,10.0,-0.961951,14.668804


In [3]:
HTML(anim.to_jshtml())

In [4]:
anim.save("medias.mp4", writer=FFMpegWriter(fps=1, bitrate=12000), dpi=150)

In [5]:
anim.save("medias.gif", writer=PillowWriter(fps=1), dpi=150)